# FastSpeech 2 ? English and Telugu
Assignment pipeline: inspect submitted models, or prepare a fresh experiment.

**Default: `MODE = "evaluate"`.** This restores the published model bundles and generates the ten required inputs without retraining. `MODE = "train"` creates a new experiment, aligns fresh audio, validates every selected feature, and stops at quality gates before full training. Existing runs are preserved.

The eight sections map to setup, English data, Telugu data, diagnostic overfit, two pilots, held-out review, and final synthesis. Extra helper scripts live in `tools/` and are described in the README. Google Drive and GPU access are supplied by your Colab session. No run in this notebook is claimed to have been executed by a documentation update.

## 1 ? Setup and explicit experiment choice

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path
from google.colab import drive
from IPython.display import Audio, display
os.environ["MPLBACKEND"] = "Agg"
drive.mount("/content/drive")
MODE = "evaluate"  # Change to "train" only for a new experiment.
assert MODE in {"evaluate", "train"}
REPO = Path("/content/FastSpeech2-English-Telugu")
RUN = Path("/content/drive/MyDrive/FastSpeech2_Reviewed_v3")
RUN.mkdir(parents=True, exist_ok=True)
def run(command, cwd=None):
    subprocess.run([str(v) for v in command], cwd=cwd or REPO, check=True)
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/SandeshAwate6055/FastSpeech2-English-Telugu.git", str(REPO)], check=True)
else:
    status = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True)
    if status.strip():
        raise RuntimeError("Checkout has edits. Preserve/commit them before updating; no reset was performed.\n" + status)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
run([sys.executable, "-m", "pip", "install", "-q", "-r", REPO / "requirements.txt"])
run([sys.executable, "-c", "import nltk; [nltk.download(n, quiet=True) for n in ['averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'cmudict']]"])
# Restore immutable metadata and checksum-verified acoustic/vocoder weights.
# If Drive presents a confirmation page, follow README manual downloads and rerun.
run([sys.executable, REPO / "tools/restore_artifacts.py", "--download"])
if MODE == "train" and not Path("/usr/local/bin/micromamba").exists():
    run(["bash", "-lc", "curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /tmp bin/micromamba && cp /tmp/bin/micromamba /usr/local/bin/micromamba"])
print("Mode:", MODE, "Run:", RUN)
print(subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

## 2 ? English: restored inference bundle or fresh data and alignment
Training uses new raw/features paths and records the resolved dataset revision. No old TextGrid-count or archive-size shortcut is used. All four source manifests are preserved. Telugu MFA is fitted on training/reserve-training audio only.

In [ ]:
if MODE == "train":
    # Svarah needs prior Hugging Face approval and HF_TOKEN in Colab Secrets.
    if "English" == "English" and not os.environ.get("HF_TOKEN"):
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    run([sys.executable, REPO / "tools/colab_pipeline.py", "prepare", "English", "--run-dir", RUN])
else:
    print("English: using published checkpoint and bundled inference metadata; dataset download is unnecessary.")

## 3 ? Telugu: restored inference bundle or fresh data and alignment
Training uses new raw/features paths and records the resolved dataset revision. No old TextGrid-count or archive-size shortcut is used. All four source manifests are preserved. Telugu MFA is fitted on training/reserve-training audio only.

In [ ]:
if MODE == "train":
    # Svarah needs prior Hugging Face approval and HF_TOKEN in Colab Secrets.
    if "Telugu" == "English" and not os.environ.get("HF_TOKEN"):
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    run([sys.executable, REPO / "tools/colab_pipeline.py", "prepare", "Telugu", "--run-dir", RUN])
else:
    print("Telugu: using published checkpoint and bundled inference metadata; dataset download is unnecessary.")

## 4 ? Vocoder and 64-clip overfit diagnostic
For a new run, inspect real-mel reconstructions for English and both Telugu voices, then compare reference-conditioned and free-running overfit probes. The diagnostic uses 64 training clips and speaker-matched held-out clips. Its statistics are inherited from the full training split; it is not the final assignment model.

In [ ]:
if MODE == "train":
    for language in ["English", "Telugu"]:
        cfg = RUN / language / "config.yaml"
        out = RUN / language / "vocoder_probe"
        run([sys.executable, REPO / "tools/probe_audio.py", "--language", language, "--config", cfg, "--output", out])
        for wav in sorted(out.glob("*.wav")):
            print(wav.name); display(Audio(str(wav)))
        run([sys.executable, REPO / "tools/colab_pipeline.py", "overfit", language, "--run-dir", RUN])
        for wav in sorted((RUN / language / "overfit/audio").glob("*.wav")):
            print(wav.name); display(Audio(str(wav)))
else:
    print("Evaluation mode: proceed to section 8 for the ten assignment outputs.")

## 5 ? English: 5,000-step held-out pilot

In [ ]:
OVERFIT_AUDIO_REVIEWED_ENGLISH = False  # Set True only after listening to section 4.
if MODE == "train":
    if not OVERFIT_AUDIO_REVIEWED_ENGLISH:
        raise RuntimeError("Review real-mel and free-running overfit speech before starting the pilot.")
    cfg = RUN / "English" / "config.yaml"
    ckpt = RUN / "English" / "checkpoints"
    steps = [int(p.name.split('.')[0]) for p in ckpt.glob("*.pth.tar") if p.name.split('.')[0].isdigit()]
    restore = max(steps, default=0)
    if restore < 5000:
        run([sys.executable, REPO / "English/train.py", "--config", cfg, "--restore_step", restore, "--total_step", 5000, "--val_step", 1000, "--save_step", 1000])
    print("Pilot ready for held-out probes; resume is checked against feature/source/config hashes.")

## 6 ? Telugu: 5,000-step held-out pilot

In [ ]:
OVERFIT_AUDIO_REVIEWED_TELUGU = False  # Set True only after listening to section 4.
if MODE == "train":
    if not OVERFIT_AUDIO_REVIEWED_TELUGU:
        raise RuntimeError("Review real-mel and free-running overfit speech before starting the pilot.")
    cfg = RUN / "Telugu" / "config.yaml"
    ckpt = RUN / "Telugu" / "checkpoints"
    steps = [int(p.name.split('.')[0]) for p in ckpt.glob("*.pth.tar") if p.name.split('.')[0].isdigit()]
    restore = max(steps, default=0)
    if restore < 5000:
        run([sys.executable, REPO / "Telugu/train.py", "--config", cfg, "--restore_step", restore, "--total_step", 5000, "--val_step", 1000, "--save_step", 1000])
    print("Pilot ready for held-out probes; resume is checked against feature/source/config hashes.")

## 7 ? Held-out checkpoint probes and optional production training
These probes use ordinary validation utterances, never the ten mandatory test sentences. Compare the saved CSVs and free-running speech. A larger step count does not guarantee better speech. Full training requires explicit listening review and retains the same feature provenance.

In [ ]:
ALLOW_FULL_RUN = False
PILOT_AUDIO_REVIEWED = False
if MODE == "train":
    for language in ["English", "Telugu"]:
        for step in [1000, 3000, 5000]:
            checkpoint = RUN / language / "checkpoints" / f"{step}.pth.tar"
            if not checkpoint.exists():
                continue
            out = RUN / language / "heldout_probe" / str(step)
            run([sys.executable, REPO / "tools/probe_audio.py", "--language", language, "--config", RUN / language / "config.yaml", "--ckpt-file", checkpoint, "--output", out])
            for wav in sorted(out.glob("*free_running.wav")):
                print(language, step, wav.name); display(Audio(str(wav)))
    if ALLOW_FULL_RUN:
        if not PILOT_AUDIO_REVIEWED:
            raise RuntimeError("Review the held-out pilot speech before full training.")
        for language in ["English", "Telugu"]:
            ckpt = RUN / language / "checkpoints"
            restore = max(int(p.name.split('.')[0]) for p in ckpt.glob("*.pth.tar") if p.name.split('.')[0].isdigit())
            if restore < 60000:
                run([sys.executable, REPO / language / "train.py", "--config", RUN / language / "config.yaml", "--restore_step", restore, "--total_step", 60000, "--val_step", 1000, "--save_step", 1000])

## 8 ? Exact test synthesis, listening review and packaging
The fixed test texts are preserved, including E3/T3 double spaces. Output goes into this run's `Results/`, preserving committed results. Select a checkpoint using held-out evidence first. Evaluation mode uses English 8k/speaker 14 and Telugu 13k/speaker 0. A new run's best validation checkpoint remains provisional until its free-running speech is reviewed.

In [ ]:
registry = json.loads((REPO / "model_artifacts.json").read_text())
tests = json.loads((REPO / "Results/test_sentences.json").read_text())
for language in ["English", "Telugu"]:
    result = RUN / "Results" / language
    if (result / "variance_predictions_summary.json").exists():
        raise RuntimeError(f"Results already exist at {result}; inspect them or select a new RUN. No overwrite performed.")
    result.mkdir(parents=True, exist_ok=True)
    if MODE == "evaluate":
        cfg = REPO / language / "config.yaml"
        checkpoint = REPO / registry[language]["checkpoint_file"]
        speaker = registry[language]["speaker_id"]
        losses = REPO / "Results" / language
    else:
        cfg = RUN / language / "config.yaml"
        checkpoint = RUN / language / "checkpoints/best_val.pth.tar"
        speaker = -1 if language == "English" else 0
        losses = RUN / language / "results"
    run([sys.executable, REPO / language / "inference.py", "--config", cfg, "--ckpt_file", checkpoint, "--speaker_id", speaker, "--result_path", result])
    summary = json.loads((result / "variance_predictions_summary.json").read_text())
    assert set(summary) == set(tests[language])
    for sid, row in summary.items():
        assert row["raw_text"] == tests[language][sid]
        assert sum(row["durations_frames"]) == row["total_frames"]
        wav = result / "test_generated_audios" / f"{sid}.wav"
        assert wav.is_file() and wav.stat().st_size > 44
        for kind in ["duration", "pitch", "energy"]:
            assert (result / "variance_adaptor_plots" / kind / f"{sid}_{kind}.png").is_file()
        print(language, sid, row["checkpoint_step"], row["speaker_id"]); display(Audio(str(wav)))
    for name in ["training_loss.png", "validation_loss.png"]:
        assert (losses / name).exists(), f"Missing {name}"
        shutil.copy2(losses / name, result / name)
archive = RUN / "Results_Deliverables_Reviewed.zip"
if archive.exists():
    raise RuntimeError("Existing deliverables ZIP preserved; choose a new RUN before packaging.")
shutil.make_archive(str(archive.with_suffix('')), 'zip', root_dir=RUN, base_dir='Results')
print("Generated ten WAVs, thirty variance plots and four loss figures:", archive)
print("Listen to every file. Record observations using Report/audio_review.html locally; export notes and rebuild the report.")